In [ ]:
# Phase 10 Gate G1, on Kaggle: five-fold heads on the cached DINOv3-L features. Rules: NOTES 2026-10-06 (registered before the cache existed).
import glob, hashlib, os, shutil, sys, time
GIT_SHA = '78ebbb4'
SRC = glob.glob('/kaggle/input/**/rsna-knee-src', recursive=True)[0]
COMP_DIR = glob.glob('/kaggle/input/**/rsna-knee-abnormality-detection', recursive=True)[0]
PKG = '/kaggle/working/knee'
os.makedirs(PKG, exist_ok=True)
for fname in os.listdir(SRC):
    if fname.endswith('.py'):
        shutil.copy(os.path.join(SRC, fname), os.path.join(PKG, fname))
sys.path.insert(0, '/kaggle/working')
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from pathlib import Path
assert torch.cuda.is_available()
print('GPU:', torch.cuda.get_device_name(0))
from knee.feature_head import FeatureHead
from knee.features import load_feature_cache
from knee.infer import LABEL_COLUMNS
from knee.metrics import macro_auc, paired_macro_auc_delta, per_label_auc
from knee.reports import blend_sources, report_group_key
from knee.train import make_folds, masked_bce_loss
print('src ok | GIT_SHA', GIT_SHA)


In [ ]:
# Folds v3 and labels. Folds are regenerated here rather than uploaded, and
# pinned by hash: the experiment discipline requires one frozen fold assignment
# across every Phase 6 run, and a silent drift would make every paired delta
# meaningless without failing anything.
import hashlib

from knee.infer import LABEL_COLUMNS
from knee.metrics import macro_auc, paired_macro_auc_delta, per_label_auc
from knee.train import Timer, evaluate, log_experiment, train_val_split
from torch.utils.data import DataLoader

train_df = pd.read_csv(f'{COMP_DIR}/train.csv')
all_uids = sorted(train_df['StudyInstanceUID'].astype(str))
assert len(all_uids) == 4407

groups = {u: report_group_key(r)
          for u, r in zip(train_df['StudyInstanceUID'].astype(str), train_df['Report'])}
folds = make_folds(all_uids, n_folds=5, seed=0, groups=groups)
FOLDS_PRIMARY_V3_SHA256 = 'f1d6ba7c341f8d2e82241cb4ddd5ca5131b0c6d2033eb6d2e43bcb597cf12868'
assert hashlib.sha256(
    '\n'.join(f'{u},{folds[u]}' for u in sorted(folds)).encode()
).hexdigest() == FOLDS_PRIMARY_V3_SHA256, 'fold assignment drifted from folds_primary_v3.csv'
n_straddle = sum(1 for _, g in pd.DataFrame(
    {'g': [groups[u] for u in all_uids], 'f': [folds[u] for u in all_uids]}
).groupby('g') if g.f.nunique() > 1)
print(f'folds v3 verified; {n_straddle} report groups straddle folds (v2 had 47)')

gold_cols = [c for c in train_df.columns if c not in ('StudyInstanceUID', 'Report')]
gold_df = train_df[train_df[gold_cols].notna().any(axis=1)]
holdout = frozenset(gold_df['StudyInstanceUID'].astype(str))
print(f'{len(holdout)} gold studies held out of both sides of every split')

# Step 3 re-baselines on the same label source Phase 5 trained on; swapping the
# label target is c0, a separate cell, so the two changes stay attributable.
pseudo = pd.read_csv(glob.glob('/kaggle/input/**/pseudo_labels_qwen3_4b.csv', recursive=True)[0])
labels_all = pseudo[['StudyInstanceUID'] + [f'score_{l}' for l in LABEL_COLUMNS]]
labels_all.columns = ['StudyInstanceUID'] + LABEL_COLUMNS

# Two frames, as Phase 5 used: soft scores are the training target, but AUC is
# scored against a binarized one -- roc_auc_score rejects a continuous y_true,
# and the soft target is a confidence, not a label.
labels_eval = labels_all.copy()
labels_eval[LABEL_COLUMNS] = (labels_all[LABEL_COLUMNS].to_numpy(dtype=float) >= 0.5).astype(float)
print(f'{len(labels_all)} pseudo-labelled studies (soft for training, binarized for scoring)')

# c0's alternative target. steven_v4 is the best single public source on the 58
# gold studies (0.8927 vs our 0.8616, paired delta +0.031 [+0.004, +0.058]).
#
# NOT the 5-source rank blend, which scored higher on gold (0.8976): percentile
# ranks are uniform by construction, so every label's target mean is exactly
# 0.500 and all prevalence information is destroyed -- Fracture would be trained
# at 0.5 against a real rate of 0.014. AUC never notices, because AUC reads
# order only. That is exactly why rank-averaging is right for combining
# predictions at submission time and wrong for building a target.
LABEL_SOURCES = {'ours_qwen3_4b': (labels_all, labels_eval)}
steven = glob.glob('/kaggle/input/**/llm_labels_v4_blend.csv', recursive=True)
if steven:
    alt = pd.read_csv(steven[0])
    alt['StudyInstanceUID'] = alt['StudyInstanceUID'].astype(str)
    alt = alt[['StudyInstanceUID'] + LABEL_COLUMNS]
    alt_eval = alt.copy()
    alt_eval[LABEL_COLUMNS] = (alt[LABEL_COLUMNS].to_numpy(dtype=float) >= 0.5).astype(float)
    LABEL_SOURCES['steven_v4'] = (alt, alt_eval)
    rate = alt[LABEL_COLUMNS].mean()
    print(f'steven_v4 loaded: {len(alt)} studies, positive rate '
          f'{rate.min():.3f}-{rate.max():.3f} (ours: '
          f'{labels_all[LABEL_COLUMNS].mean().min():.3f}-'
          f'{labels_all[LABEL_COLUMNS].mean().max():.3f})')
print('label sources available:', list(LABEL_SOURCES))

# Confidence weights, derived here rather than mounted as a CSV so the derivation
# is reproducible where it is used. Four INDEPENDENT labellings vote: ours,
# steven_v4, pilkwang_v2, lixin73. Steven ships full/v2/v4_blend but those are one
# author's successive versions and would triple-count that author's reading;
# yunusgmsoy is excluded for carrying gold values on the 58 held-out rows.
#
# CAVEAT measured on 2026-09-07 and worth re-reading before believing a positive
# result: agreement is not accuracy. Across the 12 labels, disagreement correlates
# +0.40 (n=12, p=0.20) with steven_v4's gold per-label AUC -- the wrong sign.
# Synovitis is the extreme: near-perfect source agreement (1.2% disputed) and the
# worst-labelled of the twelve. confidence_from_agreement normalises to mean 1 PER
# LABEL, which removes exactly that between-label axis, so what is being tested
# here is strictly within-label. The prior is weaker than it looks.
def _conf_source(pattern, use_conf):
    hits = glob.glob(f'/kaggle/input/**/{pattern}', recursive=True)
    if not hits:
        return None
    df = pd.read_csv(hits[0])
    df['StudyInstanceUID'] = df['StudyInstanceUID'].astype(str)
    out = pd.DataFrame({'StudyInstanceUID': df['StudyInstanceUID']})
    for lab in LABEL_COLUMNS:
        if lab not in df.columns:
            return None
        v = pd.to_numeric(df[lab], errors='coerce')
        conf_col = f'{lab}__conf'
        if use_conf and conf_col in df.columns:
            c = pd.to_numeric(df[conf_col], errors='coerce').clip(0, 1).fillna(0.5)
            v = v * c + (1 - v) * (1 - c)
        out[lab] = v
    return out.set_index('StudyInstanceUID')

_votes = [labels_all.set_index('StudyInstanceUID')[LABEL_COLUMNS]]
for _pat, _uc in (('llm_labels_v4_blend.csv', False),
                  ('report_labels_v2.csv', True),
                  ('report_labels_gpt56sol.csv', True)):
    _v = _conf_source(_pat, _uc)
    if _v is not None:
        _votes.append(_v)

# Below three voices, spread is not a meaningful agreement signal -- fail loudly
# rather than train a "weighted" arm that is silently the unweighted one.
assert len(_votes) >= 4, f'only {len(_votes)} label sources mounted; need 4'


In [ ]:
# The cache and the targets, assembled as the local harness assembles them (steven_v4 + ours + lixin73 blend; steven_v4 binarised as the truth).
OUT = Path('/kaggle/working/g1'); OUT.mkdir(parents=True, exist_ok=True)
EPOCHS = 30
dev = 'cuda'
CACHE = glob.glob('/kaggle/input/**/feat_dinov3l_336.npz', recursive=True)
assert len(CACHE) == 1, CACHE
cache = load_feature_cache(CACHE[0])
cuid = [str(u) for u in cache.uids]; row = {u: i for i, u in enumerate(cuid)}
assert cuid == sorted(all_uids), 'cache row order is not the sorted corpus order'
assert len(_votes) == 4
ours, steven, lixin = _votes[0], _votes[1], _votes[3]
blend = blend_sources([steven, ours, lixin], LABEL_COLUMNS)
sv = steven[LABEL_COLUMNS].to_numpy(float)
truth_frame = pd.DataFrame(np.where(np.isnan(sv), np.nan, (sv >= 0.5).astype(float)), index=steven.index, columns=LABEL_COLUMNS)
gold_uids = [u for u in gold_df['StudyInstanceUID'].astype(str) if u in row]
labeled = sorted(u for u in cuid if u in folds and u not in holdout)
Y = torch.tensor(blend.reindex(cuid)[LABEL_COLUMNS].to_numpy(np.float32))
T = truth_frame.reindex(labeled)[LABEL_COLUMNS].to_numpy(float)
G = gold_df.set_index(gold_df['StudyInstanceUID'].astype(str)).loc[gold_uids, LABEL_COLUMNS].to_numpy(float)
print(f'cache {cache.features.shape} | labeled {len(labeled)} | gold {len(gold_uids)}', flush=True)


In [ ]:
FEAT = torch.from_numpy(cache.features)        # fp16 [N, S, G, D] on CPU
MASK = torch.from_numpy(cache.mask)
D = FEAT.shape[-1]

class Net(nn.Module):
    def __init__(self, adapter):
        super().__init__()
        self.drop = nn.Dropout(0.2)
        self.adapter = nn.Sequential(nn.Linear(D, 512), nn.GELU(), nn.Dropout(0.3)) if adapter else nn.Identity()
        self.head = FeatureHead(512 if adapter else D)
    def forward(self, x, mask):
        return self.head(self.adapter(self.drop(x)), mask)

def batch(idx, mu, sd, windows=None):
    x = FEAT[idx].to(dev).float(); m = MASK[idx].to(dev)
    x = (x - mu) / sd * m[:, :, None, None]
    if windows == 'random':
        B, S, Gw, _ = x.shape
        pick = torch.rand(B, Gw, device=dev).argsort(1)[:, :5].sort(1).values
        x = torch.gather(x, 2, pick[:, None, :, None].expand(B, S, 5, x.shape[-1]))
    elif windows is not None:
        x = x[:, :, windows]
    return x, m

def fit_stats(idx):
    s = torch.zeros(D, device=dev); ss = torch.zeros(D, device=dev); n = 0
    for i in range(0, len(idx), 256):
        x = FEAT[idx[i:i + 256]].to(dev).float(); m = MASK[idx[i:i + 256]].to(dev)
        tok = x[m]                                  # present slots only: [n_slots_present, G, D]
        s += tok.sum((0, 1)); ss += (tok ** 2).sum((0, 1)); n += tok.shape[0] * tok.shape[1]
    mu = s / n; sd = (ss / n - mu ** 2).clamp(min=1e-6).sqrt()
    return mu[None, None, None, :], sd[None, None, None, :]

@torch.no_grad()
def predict(net, idx, mu, sd, windows):
    net.eval(); out = []
    for i in range(0, len(idx), 128):
        x, m = batch(idx[i:i + 128], mu, sd, windows)
        out.append(torch.sigmoid(net(x, m)).cpu().numpy())
    return np.concatenate(out)

FIVE = torch.tensor([0, 2, 4, 6, 8])
res = {}
for arm, adapter in (('A_linear', False), ('B_adapter', True)):
    oof9 = np.zeros((len(labeled), 12), np.float32); oof5 = np.zeros_like(oof9)
    gold9 = np.zeros((len(gold_uids), 12), np.float32); gold5 = np.zeros_like(gold9)
    pos = {u: i for i, u in enumerate(labeled)}
    gidx = torch.tensor([row[u] for u in gold_uids])
    t0 = time.time()
    for f in range(5):
        tr = [row[u] for u in labeled if folds[u] != f]; va = [u for u in labeled if folds[u] == f]
        tr = torch.tensor(tr); vai = torch.tensor([row[u] for u in va])
        torch.manual_seed(f); mu, sd = fit_stats(tr)
        net = Net(adapter).to(dev)
        opt = torch.optim.Adam(net.parameters(), lr=1e-3)
        steps = -(-len(tr) // 64)
        sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=3e-3, epochs=EPOCHS, steps_per_epoch=steps)
        for ep in range(EPOCHS):
            net.train(); perm = tr[torch.randperm(len(tr))]; tot = 0
            for i in range(0, len(perm), 64):
                b = perm[i:i + 64]
                x, m = batch(b, mu, sd, 'random')
                loss = masked_bce_loss(net(x, m), Y[b].to(dev))
                opt.zero_grad(); loss.backward(); opt.step(); sched.step(); tot += float(loss)
        for k, w in ((9, None), (5, FIVE)):
            p = predict(net, vai, mu, sd, w); g = predict(net, gidx, mu, sd, w)
            ix = [pos[u] for u in va]
            (oof9 if k == 9 else oof5)[ix] = p
            (gold9 if k == 9 else gold5)[:] += g / 5
        ok = ~np.isnan(T[[pos[u] for u in va]]).all(1)
        print(f'{arm} fold {f}: last loss {tot / steps:.4f} | val macro(9) {macro_auc(T[[pos[u] for u in va]], oof9[[pos[u] for u in va]]):.4f} | {time.time() - t0:.0f}s', flush=True)
    for name, arr in (('oof9', oof9), ('oof5', oof5), ('gold9', gold9), ('gold5', gold5)):
        np.save(OUT / f'{arm}_{name}.npy', arr)
    res[arm] = dict(oof9=oof9, oof5=oof5, gold9=gold9, gold5=gold5)

# ---- readout --------------------------------------------------------------------------------------------
np.save(OUT / 'truth.npy', T); np.save(OUT / 'gold_true.npy', G)
(OUT / 'labeled_uids.txt').write_text('\n'.join(labeled)); (OUT / 'gold_uids.txt').write_text('\n'.join(gold_uids))
print('\n==== READOUT (pooled OOF vs the fixed truth, steven_v4 binarised) ====')
for arm, r in res.items():
    print(f'{arm:10s} OOF 9 windows {macro_auc(T, r["oof9"]):.4f} | 5 windows {macro_auc(T, r["oof5"]):.4f} | '
          f'gold 9 {macro_auc(G, r["gold9"]):.4f} | 5 {macro_auc(G, r["gold5"]):.4f}')
best = max(res, key=lambda a: macro_auc(T, res[a]['oof9']))
print('arm read by the gate (higher pooled OOF at nine windows):', best)
print('per label OOF 9:', ' '.join(f'{a:.3f}' for a in per_label_auc(T, res[best]['oof9'])))
